# 05 - Interpretation (KDD stage 5)
Evaluate against the SMART objectives and write the model card.

In [ ]:
import json
import sys
from pathlib import Path

try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks" / "common.py").exists())
except StopIteration:
    raise SystemExit("notebooks/common.py not found. Run inside the repo.")
sys.path.insert(0, str(ROOT / "notebooks"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

m = json.loads((ROOT / "models" / "metrics.json").read_text())
preds = np.load(ROOT / "models" / "test_predictions.npz")
fig_dir = ROOT / "docs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

table = pd.DataFrame({k: {a: b for a, b in v.items() if a != "confusion_matrix"}
                      for k, v in m["models"].items()}).T.round(3)
print(table)

In [ ]:
TARGET = 0.80
yt = preds["y"]
print(f"Always-predict-'no flood' accuracy on the test split: {1 - yt.mean():.3f}  (why accuracy alone misleads)")
for name, v in m["models"].items():
    verdict = "meets" if v["accuracy"] >= TARGET else "MISSES"
    print(f"{name:18s} accuracy={v['accuracy']:.3f} balanced_acc={v['balanced_accuracy']:.3f} "
          f"F1={v['f1']:.3f} AUC={v['roc_auc']:.3f} -> {verdict} the {TARGET:.0%} accuracy target")
if m["data_kind"] != "real":
    print("\nWARNING: metrics come from SYNTHETIC data. They validate the pipeline, not the project objective.")

In [ ]:
fig, axes = plt.subplots(1, len(m["models"]), figsize=(4 * len(m["models"]), 3.6))
for ax, (name, v) in zip(np.atleast_1d(axes), m["models"].items()):
    cm = np.array(v["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, str(val), ha="center", va="center", fontsize=12)
    ax.set_xticks([0, 1], ["no flood", "flood"])
    ax.set_yticks([0, 1], ["no flood", "flood"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    ax.set_title(name)
fig.tight_layout()
fig.savefig(fig_dir / "confusion_matrices.png", dpi=130)

plt.figure(figsize=(5, 4))
for name in m["models"]:
    fpr, tpr, _ = roc_curve(yt, preds[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC {m['models'][name]['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.legend()
plt.tight_layout()
plt.savefig(fig_dir / "roc.png", dpi=130)

In [ ]:
p = m["primary"]
card = f"""# Model card - EWSA flood risk (hybrid CNN-LSTM)

- **Data kind:** {m['data_kind']} (dataset SHA-256: {json.dumps(m['dataset_sha256'])})
- **Intended use:** decision support for NDMC officers. Not a substitute for official SAWS warnings.
- **Architecture:** CNN over 4-channel 16x16 tiles + LSTM over a 14-day window, fused MLP head.
- **Seed:** {m['seed']}
- **Hyperparameters:** {json.dumps(m['hyperparameters'])}
- **Primary model test metrics:** {json.dumps({k: round(v, 3) for k, v in m['models'][p].items() if k != 'confusion_matrix'})}
- **Baselines:** {', '.join(k for k in m['models'] if k != p)}
- **Uncertainty:** MC-dropout std {m['uncertainty']['mc_dropout_mean_std']:.3f} (not a calibrated interval).
- **Novelty detector:** autoencoder; flags {m['novelty']['flag_rate_test']:.1%} of test rows, {m['novelty']['flag_rate_shifted']:.1%} of shifted rows.

## Limitations
- If data kind is synthetic, these numbers say nothing about real-world performance.
- Single hazard (flood) only; no disaster-type classification yet.
- Not yet loaded by the API: the live alert path uses the statistical baseline predictor.
"""
(ROOT / "models" / "model_card.md").write_text(card, encoding="utf-8")
print("wrote models/model_card.md")

In [ ]:
{
 "cells": [
  {"cell_type": "markdown", "metadata": {}, "source": "# KDD Stage 5 — Interpretation\n\nEvaluates the CNN-LSTM against the baseline on the held-out test split, checks the\n**≥80% accuracy SMART objective explicitly (pass/fail)**, and writes `models/metrics.json`,\nwhich the backend's `/models/metrics` endpoint serves to the dashboard's Model Performance page."},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "import numpy as np, json\nfrom pathlib import Path\nimport joblib\nimport tensorflow as tf\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix\n\nPROC = Path(\"../data/processed\")\nMODELS = Path(\"../models\")\nSMART_TARGET = 0.80\n\nd = np.load(PROC / \"model_ready.npz\")\nX_seq, X_tile, y = d[\"X_seq\"], d[\"X_tile\"], d[\"y\"]\ntest_idx = d[\"test_idx\"]\n\nmodel = tf.keras.models.load_model(MODELS / \"cnn_lstm.keras\")\nrf = joblib.load(MODELS / \"baseline_rf.joblib\")\nautoencoder = tf.keras.models.load_model(MODELS / \"novelty_ae.keras\")\nmeta = json.loads((MODELS / \"artefact_meta.json\").read_text())"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "def flatten_for_baseline(X):\n    return np.concatenate([X[:, -1, :], X.mean(axis=1), X.max(axis=1)], axis=1)\n\ny_test = y[test_idx]\n\nprob_cnn = model.predict({\"sequence\": X_seq[test_idx], \"tile\": X_tile[test_idx]}).ravel()\npred_cnn = (prob_cnn >= 0.5).astype(int)\n\nXb_test = flatten_for_baseline(X_seq[test_idx])\nprob_rf = rf.predict_proba(Xb_test)[:, 1]\npred_rf = (prob_rf >= 0.5).astype(int)\n\ndef metric_block(y_true, y_pred, y_prob):\n    return {\n        \"accuracy\": round(float(accuracy_score(y_true, y_pred)), 4),\n        \"precision\": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),\n        \"recall\": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),\n        \"f1\": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),\n        \"roc_auc\": round(float(roc_auc_score(y_true, y_prob)), 4),\n        \"confusion_matrix\": confusion_matrix(y_true, y_pred).tolist(),\n    }\n\ncnn_metrics = metric_block(y_test, pred_cnn, prob_cnn)\nrf_metrics = metric_block(y_test, pred_rf, prob_rf)\nprint(\"CNN-LSTM:\", cnn_metrics)\nprint(\"Baseline RF:\", rf_metrics)"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "passed = cnn_metrics[\"accuracy\"] >= SMART_TARGET\nprint(f\"SMART objective (accuracy >= {SMART_TARGET:.0%}): {'PASS' if passed else 'FAIL'} \"\n      f\"({cnn_metrics['accuracy']:.2%})\")\nassert passed, \"Model did not reach the SMART accuracy target on the held-out test set\""},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "# Qualitative OWML check: feed an out-of-envelope synthetic sample, confirm it's flagged novel\n# rather than confidently misclassified.\nextreme = X_seq[test_idx][:1].copy()\nextreme[0, :, 0] = 12.0  # rainfall feature far outside normalised training range\nrecon_err = float(np.mean((autoencoder.predict(extreme.reshape(1, -1)) - extreme.reshape(1, -1)) ** 2))\nis_novel = recon_err > meta[\"novelty_threshold\"]\nprint(f\"reconstruction error {recon_err:.4f} vs threshold {meta['novelty_threshold']:.4f} -> \"\n      f\"{'flagged as novel' if is_novel else 'NOT flagged (investigate)'}\")"},
  {"cell_type": "code", "execution_count": null, "metadata": {}, "outputs": [], "source": "metrics_out = {\n    \"dataset_version\": meta[\"dataset_version\"], \"smart_target\": SMART_TARGET,\n    \"smart_target_met\": passed,\n    \"models\": {\"cnn_lstm\": cnn_metrics, \"baseline_random_forest\": rf_metrics},\n    \"novelty_detector\": {\"threshold\": meta[\"novelty_threshold\"], \"qualitative_check_passed\": bool(is_novel)},\n    \"note\": \"Computed on a synthetic dataset (data/raw generated by 01_selection.ipynb). \"\n            \"This validates the pipeline end-to-end; it is NOT evidence of real-world skill \"\n            \"until re-run on real SAWS/Sentinel-2 data. See docs/DATA_SOURCES.md.\",\n}\n(Path(\"../models\") / \"metrics.json\").write_text(json.dumps(metrics_out, indent=2))\nmetrics_out"}
 ],
 "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}, "language_info": {"name": "python", "version": "3.10"}},
 "nbformat": 4,
 "nbformat_minor": 5
}